# EA2 — Despliegue y gobierno de una infraestructura de datos en la nube

**Big Data (ISD-25)** · Ingeniería de Software y Datos · IU Digital de Antioquia

| | |
|---|---|
| **Grupo** | *70* |
| **Integrantes** | Isabela Cuartas Vence · Juan Camilo Gomez Murillo|
| **Caso de estudio** | *Wanderbricks* |
| **Fecha de entrega** | domingo 6 de septiembre |
| **🎥 Enlace al video** | *(pegar aquí — 6 a 9 minutos, mínimo 3 minutos por integrante)* |

> ⚠️ **Antes de entregar:** verificar que el enlace del video abra desde una cuenta distinta a la propia.
> Un enlace inaccesible se califica como no entregado.

---
## 1. Contexto y problema

*Qué necesidad de infraestructura plantea el caso y qué debe soportar el entorno.*

En la EA1 construimos una base de datos analítica para Wanderbricks sobre Databricks, 
con capas bronce y plata para usuarios, reservas, actualizaciones de reservas y países. 
Esa evidencia resolvió el *qué* (modelo de datos, transformaciones, consultas), pero 
dejó sin resolver el *dónde y bajo qué reglas* vive esa información: quién puede leer 
la capa plata, quién puede escribir en bronce, cómo se automatiza la actualización 
diaria de las tablas sin intervención manual, y qué pasaría si Wanderbricks tuviera 
que auditar quién accedió a los datos de un usuario específico.

Esta necesidad no es hipotética: un marketplace real como Wanderbricks tiene equipos 
con distintos niveles de confianza sobre los mismos datos. El equipo de analítica de 
negocio necesita consultar reservas y países para reportes, pero no debería poder 
alterar las tablas fuente. El equipo de ingeniería de datos necesita escribir en 
bronce y plata, pero no necesariamente tiene por qué gestionar permisos de otros 
usuarios. Un administrador necesita visibilidad total para auditoría y gobierno. Sin 
un entorno que separe estas responsabilidades mediante catálogos, esquemas y permisos 
explícitos (GRANT), cualquier persona con acceso al workspace podría, en teoría, 
modificar o eliminar las tablas plata que alimentan los reportes de cancelación por 
país que construimos en la EA1.

Por eso, el entorno que se despliega en esta evidencia debe soportar tres cosas 
además del almacenamiento: **gobierno de acceso** (permisos diferenciados por capa y 
por rol), **automatización** (que la actualización de bronce a plata no dependa de 
que alguien ejecute el notebook manualmente cada vez que llegan datos nuevos) y 
**trazabilidad** (poder responder, vía linaje, de dónde viene cada tabla y qué la 
transforma). Adicionalmente, como Databricks es una plataforma gestionada (PaaS), el 
entorno debe dejar explícito qué responsabilidades asume el proveedor (aprovisionamiento 
de clústeres, parcheo, disponibilidad del metastore) y cuáles siguen siendo nuestras 
(diseño de esquemas, políticas de acceso, definición de jobs), como base para la 
comparación con IaaS que se pide más adelante.

---
## 2. Descripción de los datos

*Qué va a vivir en esta infraestructura: volumen esperado, frecuencia de actualización
y quién la va a consumir.*

La infraestructura que se despliega en esta evidencia opera sobre el mismo catálogo 
construido en la EA1: `bigdata_grupo70.wanderbricks`, con las tablas bronce y plata 
derivadas de `samples.wanderbricks.{users, bookings, booking_updates, countries}`. 
El volumen que debe soportar el entorno es el siguiente:

| Tabla (capa) | Filas | Rol en la infraestructura |
|---|---:|---|
| `bronze_users` | 124,509 | Insumo crudo, solo lectura para la mayoría de roles |
| `bronze_bookings` | 72,247 | Insumo crudo |
| `bronze_booking_updates` | 83,068 | Insumo crudo, el de mayor tasa de crecimiento (más filas que reservas) |
| `bronze_countries` | 168 | Tabla de dimensión, cambia con muy poca frecuencia |
| `silver_bookings_estado_actual` | 72,246 | Tabla de consumo analítico principal |
| `silver_bookings_anidado` | 72,246 | Tabla de consumo con estructura anidada |

En términos de **volumen**, ninguna tabla individual supera los 125,000 registros, lo 
cual en Databricks Free Edition no representa presión de cómputo relevante; el 
volumen relevante para el diseño de infraestructura no es el tamaño actual, sino el 
**patrón de crecimiento**: `booking_updates` ya tiene más filas que `bookings` 
(83,068 vs. 72,247), lo que confirma que es una tabla de eventos que crece con cada 
interacción del usuario sobre una reserva existente, no solo con reservas nuevas. Esto 
importa para el diseño del Job de automatización: la ingesta de `booking_updates` debe 
tratarse como incremental (nuevos eventos que llegan constantemente), mientras que 
`countries` es prácticamente estática y no necesita reprocesarse con la misma 
frecuencia que las demás.

En cuanto a **frecuencia de actualización esperada**, se distinguen tres patrones 
distintos, relevantes para decidir cómo programar el Job de la sección 6:
- `bronze_countries`: actualización esporádica (nuevos países o cambios de continente son infrecuentes).
- `bronze_users`: actualización diaria (altas de nuevos usuarios).
- `bronze_bookings` y `bronze_booking_updates`: actualización de alta frecuencia, ya que cada acción de un usuario (crear, modificar o cancelar una reserva) genera una fila nueva.

En cuanto a **quién consume esta infraestructura**, se identifican tres perfiles, que 
son la base directa de la matriz de roles de la sección 4:
- **Analista de negocio**: consume las tablas plata (`silver_bookings_estado_actual`, `silver_bookings_anidado`) para generar los reportes que se construyeron en la EA1 (tasa de cancelación por país, volumen por continente). No necesita ni debería tener acceso de escritura sobre bronce ni plata.
- **Ingeniero de datos**: es quien ejecuta o mantiene el pipeline bronce → plata, por lo que necesita permisos de escritura sobre ambas capas, pero no necesariamente permisos de administración de otros usuarios.
- **Administrador**: gestiona catálogos, esquemas, permisos de los otros dos roles y supervisa el Job de automatización; requiere visibilidad y control total sobre el entorno.

Esta segmentación por rol y por capa es la que se traduce, en la sección 3, en la 
organización de catálogos/esquemas, y en la sección 4, en los `GRANT` ejecutados 
sobre objetos concretos de este mismo catálogo. 

---
## 3. Decisiones de diseño y justificación
### 3.1 Diagrama de la arquitectura

*Fuentes → ingesta → almacenamiento → procesamiento → consumo.
Señalar explícitamente qué capa administra el proveedor y cuál el equipo.
Insertar la imagen o usar un diagrama en texto.*

![Diagrama Ea2.png](./Diagrama Ea2.png "Diagrama Ea2.png")

### 3.2 Matriz de roles

*Qué puede hacer cada rol sobre cada capa en un entorno real.*

| Rol | Bronce | Plata | Oro |
|---|---|---|---|
| Analista | No tiene acceso | Lectura | Lectura |
| Ingeniero de datos | Lectura y escritura | Lectura y escritura | Lectura y Escritura |
| Administrador | Control total | Control total | Control total |

### 3.3 Especificación del equivalente IaaS

*Se diseña, no se implementa. Máquinas y dimensionamiento, sistema operativo,
software a instalar, red, almacenamiento, y estimación del esfuerzo de puesta
en marcha y de operación.*

______________________________________________________________________________________________________________________

### Maquinas y dimensionamiento
Para soportar el procesamiento distribuido en memoria de las capas medallion

Nodo maestro / Driver: AWS EC2 t3.xlarge con 4 vCPUs y 16 GB de RAM. 
Encargado de la coordinacion del cluster y plan de ejecucion

Nodos de trabajo / Workers: AWS Ec2 r5.xlarge con 4 vCPUs2 y 32 GB de RAM cada uno.
Optimizados en memoria para tareas sofisticadas de PySpark.

### Sistema operativo
La distribucion del sistema operativo se basa en Ubuntu Server 22.04 LTS instalado en cada uno de los nodos del cluster y teniendo en cuenta el mantenimiento de sistema, implica la configuracion manual de repositorios, la aplicacion de parches de seguridad de forma regular, la gestion de usuarios SSH y el endurecimiento del sistema operativo.

### Software
Entorno Spark: Apache Spark 3.4 utilizando Python 3.10 y Java OpenJDK 11 en cada uno de los nodos.

Administrador de recursos: Apache YARN para la gestion de recursos computacionales.

Formato Delta: Bibliotecas Delta-Core incluidas en el classpath para el soporte de transacciones ACID.

Catalogo / Metastore: Instancia especifica de PostgreSQL para operar el Hive Metastore.

Orquestador: Servidor autonomo utilizando Apache Airflow para la programacion y ejecucion de pipelines interdependientes.

Gobierno: Apache Ranger u OpenLineage para la administracion de permisos y trazabilidad.

### Red
VPC Privada: Implementancion del cluster en una red virtual privada que incluye subredes privadas para los nodos del trabajo.

Grupos de seguridad: Normas de firewall para limitar el trafico con los puertos 7077, 8080, 5432 y el puerto 22 restringido a Bastion Host/VPN.

Gateway: NAT Gateway para salida controlada a internet.

### Almacenamiento
Data Lake: Contenedores de almacenamiento de objetos organizados en tres carpetas: Bronce, Plata y Oro.

Persistencia local: Discos ESB / Discos administrados de 100 GB por nodo para el almacenamiento temporal de spills de spark y de registros.

### Estimacion del esfuerzo de puesta en marcha y operacion
Puesta en marcha inicial 80 a 120 horas: Incorporar la elaboracion de scripts Terraform/Ansible, la configuracion de red, la optimizacion de Spark, la instalacion de YARN, Hive Metastore y Airflow.

Operacion y mantenimiento continuo 20 a 30 horas: Incluye la actualizacion del sistema operativo, la supervision de nodos inactivos, el ajuste manual del escalado de maquinas y la solucion de problemas en el cluster.

### 3.4 Comparación IaaS / PaaS / SaaS

| Criterio | IaaS | PaaS | SaaS |
|---|---|---|---|
| Control | Control completo sobre la infraestructura, sistema operativo, configuracion de red y elementos del cluster Spark | Dominio completo del codigo PySpark/SQL asi como los datos, catalogos y esquemas, la plataforma oculta la infraestructura subyacente | Unicamente se supervisan los informes, la informacion ingresada y los usuarios, no es posible alterar el motor renderizado ni la logica interna |
| Tiempo hasta el primer resultado | Se requiere la implementacion de redes, la provision de nodos, la instalacion de Java/Spark/Airflow y la configuracion de dependencias antes de iniciar el procesamiento del primer dato | El entorno de Spark y la integracion con Unity Catalog estan disponibles, solo es necesario crear el Notebook y ejecutar el codigo | Conexion directa a fuentes o tablas transformadas para crear tableros visuales al instante |
| Esfuerzo operativo | El equipo se encarga de actualizar el sistema operativo, mantener el Hive Metastore, gestionar las fallas de los nodos y configurar la seguridad de la red | La plataforma administra el autoescalado y los parches del motor, la alta disponibilidad y el mantenimiento de los clusteres | El proveedor asume la totalidad de la infraestructura, la disponibilidad y las actualizaciones de la aplicacion |
| Costo | Compensacion por horas VMs activas + elevado costo en horas/ingenieros DevOps/Platform | Pago por DBUs solo durante el tiempo de ejecucion de los trabajos, lo que disminuye los costos por inactividad | Tarifa fija por licencia, ya sea mensual o anual por cada usuario |
| Escalabilidad | Necesita scripts personalizados para el autoescalado a nivel de infraestructura y un rebalanceo manual en YARN/Kubernetes | Escalado de nodos de manera automatica en segundos, dependiendo de la carga del procesamiento de la consulta o trabajo | Escalabilidad sin complicaciones para el usuario en funcion de la cantidad de consultas recurrentes |
| Gobierno | Es necesario integrar y gestionar manualmente herramientas de terceros como Apache Ranger u OpenLineage | Administracion centralizada en permisos a traves de capas, roles y linaje de datos automaticos con Unity Catalog | Administracion de permisos a nivel de filas/columnas dentro del informe o area de trabajo, no sobre las capas de datos sin procesar |

**Conclusión:** *cuál conviene a este caso y por qué.*

El analisis comparativo demuestra que la combinacion de PaaS + SaaS constituye el equilibrio perfecto para el proyecto.
Un modelo laaS requiere un esfuerzo operativo considerable y SaaS no es suficiente por si solo la transformacion de datos en bruto, mientras que PaaS facilita la centralizacion del procesamiento intensivo de las capas medallion con autoescalado y gobernanza nativa de esta forma, el equipo concentra el 100% del esfuerzo en entrega valor analitico sin sobrecostos de infraestructura.

---
## 4. Implementación
### 4.1 Organización del entorno

In [0]:
# TODO: catálogos, esquemas y volúmenes organizados con criterio

CATALOGO = "bigdata_grupo70"
ESQUEMA_BRONCE_PLATA = "wanderbricks"   
ESQUEMA_ORO = "gold"                    
VOLUMEN = "datos_crudos"                 

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOGO}.{ESQUEMA_ORO}")
spark.sql(f"USE CATALOG {CATALOGO}")

# --- Tabla oro: agregado de negocio listo para consumo, sin lógica adicional ---
# Reutilizamos la consulta 1 de la EA1 (tasa de cancelación por país) como tabla oro:
# es exactamente el tipo de resultado que un analista consulta directo, sin tener
# que rehacer el join ni la agregación cada vez.

from pyspark.sql import functions as F

df_bookings = spark.table(f"{CATALOGO}.{ESQUEMA_BRONCE_PLATA}.silver_bookings_estado_actual")
df_users = spark.table(f"{CATALOGO}.{ESQUEMA_BRONCE_PLATA}.silver_users")
df_countries = spark.table(f"{CATALOGO}.{ESQUEMA_BRONCE_PLATA}.silver_countries")

gold_tasa_cancelacion_pais = (df_bookings
    .join(df_users, on="user_id")
    .join(
        df_countries,
        on=F.trim(F.lower(df_users["country"])) == F.trim(F.lower(df_countries["country"])),
        how="inner"
    )
    .groupBy(df_countries["country"], df_countries["continent"])
    .agg(
        F.count("*").alias("total_reservas"),
        F.round(
            100.0 * F.sum(F.when(F.col("status_vigente").isin("cancelled", "expired"), 1).otherwise(0))
            / F.count("*"), 2
        ).alias("tasa_problema_pct")
    )
    .filter(F.col("total_reservas") >= 30)
)

(gold_tasa_cancelacion_pais.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{CATALOGO}.{ESQUEMA_ORO}.gold_tasa_cancelacion_pais"))

print(f"✔ Esquemas: {CATALOGO}.{ESQUEMA_BRONCE_PLATA} (bronce+plata), {CATALOGO}.{ESQUEMA_ORO} (oro)")
spark.sql(f"SHOW TABLES IN {CATALOGO}.{ESQUEMA_ORO}").show()

Se mantuvo el esquema wanderbricks para bronce y plata (continuidad con la EA1) 
y se creó un esquema gold separado para la capa de consumo analítico. Esta 
separación por esquema, y no solo por prefijo de nombre, permite aplicar permisos 
de Unity Catalog a nivel de esquema completo: todo objeto que se cree a futuro en 
gold hereda automáticamente el permiso de solo lectura otorgado al rol analista, 
sin tener que repetir el GRANT por cada tabla nueva.

### 4.2 Permisos

*Al menos dos sentencias GRANT con niveles distintos sobre objetos distintos.*

In [0]:
# GRANT 1 — a quién, qué y por qué:
# TODO
spark.sql(f"""
    GRANT SELECT ON SCHEMA {CATALOGO}.{ESQUEMA_ORO} TO `account users`
""")

# GRANT 2 — a quién, qué y por qué:
# TODO
spark.sql(f"""
    GRANT SELECT, MODIFY ON SCHEMA {CATALOGO}.{ESQUEMA_BRONCE_PLATA} TO `isabela.cuartas@est.iudigital.edu.co`
""")

# display(spark.sql(f"SHOW GRANTS ON TABLE {TABLA}"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.{ESQUEMA_ORO}"))
display(spark.sql(f"SHOW GRANTS ON SCHEMA {CATALOGO}.{ESQUEMA_BRONCE_PLATA}"))

Se asignaron dos niveles de permisos para simular la matriz de roles:

**Esquema gold:** se otorgó SELECT al principal account users. Representa al rol analista, que solo puede consultar resultados agregados, sin acceder ni modificar las capas bronce y plata.

**Esquema wanderbricks**: se otorgaron SELECT, MODIFY a un usuario específico. Representa al ingeniero de datos, quien puede consultar y modificar las capas bronce y plata para mantener el pipeline, pero no administrar permisos de otros usuarios.

Debido a que Databricks Free Edition no permite crear grupos personalizados, se asignaron los permisos al principal disponible más cercano y se documentó la correspondencia entre cada rol y su principal.

### 4.3 Linaje

*Evidenciar el linaje de una tabla desde el Catalog Explorer. Insertar la captura.*
![Captura de pantalla 2026-09-16 094454.png](./Captura de pantalla 2026-09-16 094454.png "Captura de pantalla 2026-09-16 094454.png")
![Captura de pantalla 2026-09-16 094524.png](./Captura de pantalla 2026-09-16 094524.png "Captura de pantalla 2026-09-16 094524.png")
![Captura de pantalla 2026-09-16 094554.png](./Captura de pantalla 2026-09-16 094554.png "Captura de pantalla 2026-09-16 094554.png")
![Captura de pantalla 2026-09-16 094618.png](./Captura de pantalla 2026-09-16 094618.png "Captura de pantalla 2026-09-16 094618.png")
![Captura de pantalla 2026-09-16 094637.png](./Captura de pantalla 2026-09-16 094637.png "Captura de pantalla 2026-09-16 094637.png")

El linaje muestra toda la ruta de los datos:

samples.wanderbricks -> bronze_ -> silver_ -> gold_tasa_cancelacion_pais

Esto coincide con la arquitectura de la sección 3.1 y demuestra que los datos pueden rastrearse desde su origen hasta el resultado final.Databricks genera este linaje automáticamente a partir del código ejecutado en las secciones 4.2–4.4 de la EA1.

### 4.4 Automatización

*Un Job con al menos dos tareas encadenadas y una programación definida.
Insertar la captura de una ejecución exitosa e indicar el identificador del Job.*

**Job ID:** 366792536094073 (`wanderbricks_bronce_plata_oro`)  
**Run ID de la ejecución exitosa:** 739232371113011

![Captura de pantalla 2026-09-17 102812.png](./Captura de pantalla 2026-09-17 102812.png "Captura de pantalla 2026-09-17 102812.png")
![Captura de pantalla 2026-09-17 102642.png](./Captura de pantalla 2026-09-17 102642.png "Captura de pantalla 2026-09-17 102642.png")

El Job encadena 2 tareas: `ingesta_bronce_plata` (notebook `EA1_plantilla`, 
15m 37s) y `construir_oro` (notebook `EA2_plantilla`, 15s), con dependencia 
explícita de la segunda sobre la primera. La ejecución completa tomó 15m 53s, 
procesando 88 queries, 4,333,304 filas leídas y 778,226 filas escritas.

**Causa de las 2 ejecuciones fallidas iniciales (`RunExecutionError`):**

La celda de time travel de la sección 4.5 (heredada del notebook de la EA1) 
usaba `versionAsOf 0` para consultar el estado original de 
`silver_bookings_estado_actual`. Esa versión 0 se creó el 31 de agosto de 2026 
— más de 17 días antes de esta ejecución del Job. Delta Lake tiene una propiedad 
`delta.deletedFileRetentionDuration` con valor predeterminado de 168 horas 
(7 días): pasado ese tiempo, el proceso de limpieza (`VACUUM`) elimina físicamente 
los archivos de datos de versiones antiguas, aunque el historial de metadatos en 
`DESCRIBE HISTORY` los siga listando. Al intentar leer `versionAsOf 0`, los 
archivos ya no existían, y la consulta fallaba con un error de ejecución.

**Solución aplicada:** en vez de fijar `versionAsOf 0` como número literal, la 
celda ahora consulta `DESCRIBE HISTORY` dinámicamente para encontrar la versión 
más reciente de tipo `CREATE OR REPLACE TABLE AS SELECT` (en esta ejecución, la 
versión 24 — la escritura generada por la propia corrida del Job), que representa 
el estado de la tabla justo antes del `UPDATE` de esa misma ejecución. El 
resultado confirma que el time travel sigue funcionando correctamente: antes del 
`UPDATE` (versión 24) había 1,377 reservas en `pending`; después del `UPDATE`, 
esas mismas 1,377 filas quedaron como `expired`, sin alterar las demás categorías.

Este ajuste es además más robusto para producción: al no depender de un número 
de versión codificado a mano, la celda funciona en cualquier ejecución futura del 
Job, sin importar cuántas veces se haya reconstruido la tabla desde entonces — 
algo que un pipeline automatizado y recurrente necesita por definición, ya que 
cada corrida programada genera una nueva versión "0 lógica" que las anteriores 
no pueden seguir usando como referencia fija.



In [0]:
# TODO: código de las tareas que ejecuta el Job
# -----------------------------------------------------------------
# 4.4 Automatización — código de las tareas que ejecuta el Job
# -----------------------------------------------------------------
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
JOB_ID = 366792536094073

job = w.jobs.get(JOB_ID)

print(f"Job: {job.settings.name}")
print(f"Job ID: {job.job_id}\n")

for tarea in job.settings.tasks:
    print(f"Tarea: {tarea.task_key}")
    print(f"  Notebook: {tarea.notebook_task.notebook_path}")
    if tarea.depends_on:
        deps = [d.task_key for d in tarea.depends_on]
        print(f"  Depende de: {deps}")
    else:
        print("  Depende de: (ninguna, es la primera tarea)")
    print()

# Programación (schedule) configurada
if job.settings.schedule:
    print(f"Schedule: {job.settings.schedule.quartz_cron_expression}")
    print(f"Zona horaria: {job.settings.schedule.timezone_id}")
    print(f"Pausado: {job.settings.schedule.pause_status}")

---
## 5. Resultados

---
## 6. Conclusiones

*Qué funcionó, qué no funcionó y qué harían distinto si empezaran de nuevo.
Las conclusiones deben derivarse de los resultados mostrados arriba, no de expectativas generales.*

---
## 7. Reparto del trabajo y uso de IA

| Integrante | De qué se encargó | Qué sustenta en el video |
|---|---|---|
| | | |
| | | |
| | | |

**Uso de asistentes de IA:** *(indicar en qué partes se usó el Databricks Assistant u otra
herramienta. Está permitido; lo que se evalúa es que cada integrante pueda explicar
cualquier línea del código en el video.)*

> Este reparto debe coincidir con lo que cada persona demuestra en el video y con el
> historial de commits del repositorio. Las tres fuentes se contrastan al calificar.

---
## 📹 Preguntas obligatorias de sustentación

Cada integrante responde estas tres preguntas en su intervención del video:

1. ¿Qué parte de esta arquitectura administra el proveedor y cuál administran ustedes?
2. Muestre un GRANT que ejecutó y explique a quién le está dando qué, y por qué.
3. Si tuvieran que montar esto sobre máquinas virtuales, ¿qué sería lo primero que se les complicaría?

---
## ✅ Antes de entregar

- [ ] El diagrama de arquitectura está incluido y descrito
- [ ] Los dos GRANT están ejecutados y el SHOW GRANTS muestra el resultado
- [ ] El Job tiene dos o más tareas, está programado y hay evidencia de ejecución
- [ ] La comparación IaaS/PaaS/SaaS termina en una conclusión, no en una tabla suelta
- [ ] El enlace del video está en la portada y abre desde otra cuenta
- [ ] Todo confirmado en /ea2 y el HTML subido a Canvas